# Testing LLM APIs
Welcome to this practical session. In this notebook, we will explore how to interact with the production OpenAI API, manipulate hyperparameters like temperature, and build automated testing assertions for validating structured responses.

In [22]:
import os
import os.path as osp

from pathlib import Path
from pprint import pprint

import sys

root = Path.cwd().parent 
if str(root) not in sys.path:    
    sys.path.append(str(root))

In [23]:
from src.config import CFG

## Installation & Environment Setup
First, we need to install the official OpenAI SDK and configure our secure API token environment variable.

In [24]:
import time
import os
import json
from openai import OpenAI
import os 

# Instruct students to input their real OpenAI API key

# Initialize the standard production client
client = OpenAI(api_key=CFG.API_KEY)

print("OpenAI Production Client Initialized!")

OpenAI Production Client Initialized!


In [25]:
from pprint import pprint

## Managing Hyperparameters: Temperature Determinism
Let's see how setting a low temperature (highly deterministic) vs. a high temperature (highly creative) impacts raw API output variability.

In [45]:
prompt = "Qu'est-ce qu'un agent IA? réponds en quelques lignes"

outputs = []
print("--- Testing Deterministic Output (temperature=0.0) ---")
for i in range(2):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.2
    )
    print(f"Run {i+1}:\n")
    rep = response.choices[0].message.content.strip()
    pprint(response)
    outputs.append(response)
    print()

--- Testing Deterministic Output (temperature=0.0) ---
Run 1:

ChatCompletion(id='chatcmpl-EH9t1pTIoabjYU6uGvwRLtNgxF56B', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Un agent IA, ou agent d'intelligence artificielle, est un système capable de percevoir son environnement, de traiter des informations et de prendre des décisions autonomes pour accomplir des tâches spécifiques. Il utilise des algorithmes d'apprentissage, des règles logiques ou des modèles statistiques pour analyser des données et interagir avec les utilisateurs ou d'autres systèmes. Ces agents peuvent être utilisés dans divers domaines, tels que les assistants virtuels, les systèmes de recommandation et les robots autonomes.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1787758287, model='gpt-4o-mini-2024-07-18', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fi

In [47]:
response.choices[0].message

ChatCompletionMessage(content="Un agent IA (intelligence artificielle) est un système ou un programme informatique capable d'effectuer des tâches de manière autonome, en imitant des fonctions cognitives humaines telles que l'apprentissage, la perception, le raisonnement et la prise de décision. Ces agents peuvent interagir avec leur environnement et s'adapter à des situations nouvelles en utilisant des algorithmes d'apprentissage automatique, des réseaux de neurones ou d'autres techniques avancées. Ils sont utilisés dans divers domaines, allant des assistants virtuels aux véhicules autonomes.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

In [27]:
from src.utils import calc_similarity

In [28]:
calc_similarity(outputs[0], outputs[1])

TF-IDF Cosine Similarity: 0.5560


np.float64(0.5560020950369573)

In [29]:
print("\n--- Testing Creative Output (temperature=1.2) ---")
outputs = []

for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.2
    )
    print(f"Run {i+1}:\n")
    response = response.choices[0].message.content.strip()
    pprint(response)
    outputs.append(response)
    print()


--- Testing Creative Output (temperature=1.2) ---
Run 1:

('Un agent IA est un système informatique conçu pour percevoir son '
 'environnement, traiter des informations, et prendre des décisions autonomes '
 'ou semi-autonomes pour accomplir des tâches spécifiques. Il utilise des '
 "algorithmes d'apprentissage automatique, des règles prédéfinies et des "
 'techniques de traitement du langage naturel pour interagir avec les '
 "utilisateurs ou d'autres systèmes. Les agents IA peuvent être retrouvés dans "
 "divers domaines, comme l'assistance virtuelle, la robotique, la "
 "recommandation de contenu et bien d'autres applications.")

Run 2:

("Un agent IA, ou agent d'intelligence artificielle, est un système logiciel "
 'ou un modèle qui utilise des algorithmes pour percevoir son environnement, '
 'agir de manière autonome ou semi-autonome, et prendre des décisions basées '
 'sur des données. Ces agents peuvent être programmés pour exécuter des tâches '
 'spécifiques, comme répondre à 

In [30]:
calc_similarity(outputs[0], outputs[1])

TF-IDF Cosine Similarity: 0.5882


np.float64(0.5882115247882302)

In [ ]:
response = client.chat.completions.create(
        model="gpt-5.4-mini", # Ce modèle n'est pas à utiliser avec le paramètre température
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        reasoning_effort="low"
    )

response = response.choices[0].message.content.strip()
print(response)


BadRequestError: Error code: 400 - {'error': {'message': "Unsupported value: 'temperature' does not support 0.3 with this model. Only the default (1) value is supported.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': 'unsupported_value'}}

In [32]:
prompt = "Qu'est-ce qu'un agent IA? réponds avec 300 mots"

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt}],
    reasoning_effort="low",
    stream=True  # Enables streaming
)

for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

print() 

Un **agent IA** est un système d’intelligence artificielle conçu pour **percevoir son environnement, prendre des décisions et agir** afin d’atteindre un objectif donné. Contrairement à un simple programme qui exécute une suite d’instructions fixes, un agent IA peut souvent **s’adapter** à des situations nouvelles, analyser des informations, choisir une action et parfois apprendre de ses erreurs.

On peut le voir comme un **assistant autonome**. Par exemple, un agent IA peut répondre à des emails, réserver un rendez-vous, surveiller un réseau informatique, recommander des produits, ou encore aider à conduire une voiture. Dans tous les cas, il suit une logique simple : **observer, raisonner, agir**.

Un agent IA peut être plus ou moins sophistiqué. Certains sont très simples : ils réagissent à des règles préprogrammées, comme un chatbot basique. D’autres sont plus avancés : ils utilisent le machine learning, le traitement du langage naturel, la vision par ordinateur ou d’autres technique